# <font color="#418FDE" size="6.5" uppercase>**Transfer mit Keras**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Erklären AlexNet-Ideen wie ReLU, größere Filter, Pooling und Dropout anhand kleiner Modelle. 
- Nutzen MobileNetV2 als CPU-freundlichen eingefrorenen Merkmalsextraktor. 
- Vergleichen Transfer-Learning-Ergebnisse mit kleinen selbst trainierten CNNs. 


## **1. AlexNet Ideen**

### **1.1. AlexNet Architekturideen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_01_01.jpg?v=1787659461" width="250">



>* AlexNet zeigte die Stärke tiefer CNNs
>* ReLU macht Training einfacher und Merkmale klarer

>* Große Filter erfassen grobe Bildstrukturen.
>* Pooling verdichtet Merkmale und toleriert Verschiebungen.

>* Dropout verhindert Abhängigkeit von einzelnen Neuronen
>* AlexNet-Ideen fördern robuste CNN-Architekturen



In [ ]:
#@title Python-Code - AlexNet Architekturideen

# Dieses Beispiel zeigt zentrale AlexNet Architekturideen.
# ReLU, Pooling und Dropout werden sichtbar verglichen.
# Die Ausgabe zeigt Formen und Aktivierungsanteile.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Wir erzeugen ein kleines synthetisches Graustufenbild.
image = np.zeros((1, 16, 16, 1), dtype=np.float32)
image[:, 4:12, 4:12, 0] = 1.0

# Ein großer Filter betrachtet mehr Kontext gleichzeitig.
large_filter = np.ones((7, 7, 1, 1), dtype=np.float32) / 49.0
conv = tf.nn.conv2d(image, large_filter, strides=1, padding="SAME")

# ReLU lässt positive Hinweise durch und entfernt negative Werte.
shifted_conv = conv - 0.35
relu_map = tf.nn.relu(shifted_conv)

# Max-Pooling verdichtet räumliche Information robust.
pooled_map = tf.nn.max_pool2d(
    relu_map,
    ksize=2,
    strides=2,
    padding="VALID",
)

# Dropout deaktiviert beim Training zufällige Aktivierungen.
tf.random.set_seed(42)
dropout_layer = tf.keras.layers.Dropout(rate=0.5)
dropped_map = dropout_layer(pooled_map, training=True)

# Wir prüfen die erwarteten kleinen Tensorformen.
if pooled_map.shape != (1, 8, 8, 1):
    raise ValueError("Die Pooling-Form ist unerwartet.")

active_before = np.mean(pooled_map.numpy() > 0)
active_after = np.mean(dropped_map.numpy() > 0)

print("AlexNet-Idee: großer Filter -> ReLU -> Pooling -> Dropout")
print(f"Eingabeform: {image.shape}, nach Pooling: {pooled_map.shape}")
print(f"Aktive Werte vor Dropout: {active_before:.2f}")
print(f"Aktive Werte nach Dropout: {active_after:.2f}")

# Die Grafik zeigt die verdichtete Merkmalskarte nach Dropout.
fig, ax = plt.subplots(figsize=(5, 4))
shown_map = dropped_map.numpy()[0, :, :, 0]
image_plot = ax.imshow(shown_map, cmap="viridis")

ax.set_title("Merkmalskarte nach ReLU, Pooling und Dropout")
ax.set_xlabel("Breite der Merkmalskarte")
ax.set_ylabel("Höhe der Merkmalskarte")
fig.colorbar(image_plot, ax=ax, label="Aktivierung")

plt.show()



### **1.2. Tensorformen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_01_02.jpg?v=1787659457" width="250">



>* Bilder sind Tensoren aus Höhe, Breite, Kanälen
>* Filter erzeugen neue Merkmalskarten und Tensor-Tiefe

>* Große Filter erfassen größere Bildmuster
>* Padding und Pooling verändern räumliche Tensorgrößen

>* ReLU und Dropout ändern Werte, nicht Formen
>* Flatten verbindet Merkmalskarten mit Entscheidungen



In [ ]:
#@title Python-Code - Tensorformen verstehen

# Dieses Beispiel zeigt Tensorformen in kleinen CNNs.
# Faltung, ReLU, Pooling und Dropout werden verglichen.
# Die Ausgabe macht Formänderungen direkt sichtbar.

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Wir verwenden ein kleines synthetisches RGB-Bild.
tf.keras.utils.set_random_seed(42)

rng = np.random.default_rng(42)
image_batch = rng.random((1, 32, 32, 3), dtype=np.float32)

# Jede Schicht steht für eine typische AlexNet-Idee.
model = keras.Sequential(
    [
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(8, kernel_size=5, padding="valid", name="conv_5x5"),
        layers.ReLU(name="relu"),
        layers.MaxPooling2D(pool_size=2, name="max_pool"),
        layers.Dropout(0.5, name="dropout"),
        layers.Flatten(name="flatten"),
    ]
)

# Wir prüfen die erwartete Eingabeform vor dem Durchlauf.
if image_batch.shape != (1, 32, 32, 3):
    raise ValueError("Die Eingabe muss die Form Batch, Höhe, Breite, Kanäle haben.")

# Nun verfolgen wir die Tensorform nach jeder Schicht.
current_tensor = image_batch
print("Start: Batch=1, Höhe=32, Breite=32, Kanäle=3")

for layer in model.layers:
    current_tensor = layer(current_tensor, training=True)
    shape_text = " x ".join(str(value) for value in current_tensor.shape)
    print(f"{layer.name}: {shape_text}")

# Dropout ändert Werte, aber nicht die Tensorform.
dropout_layer = model.get_layer("dropout")
pooled_tensor = model.get_layer("max_pool")(
    model.get_layer("relu")(model.get_layer("conv_5x5")(image_batch))
)

dropped_tensor = dropout_layer(pooled_tensor, training=True)
zero_share = np.mean(np.asarray(dropped_tensor) == 0.0)

print(f"Dropout-Nullanteil im Training: {zero_share:.2f}")



### **1.3. Rechenaufwand verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_01_03.jpg?v=1787659459" width="250">



>* Größere CNNs brauchen deutlich mehr Rechenleistung
>* Architektur muss zur Hardware passen

>* Pooling verkleinert Merkmalskarten und spart Rechenzeit
>* Wichtige Muster bleiben trotz kleiner Verschiebungen erhalten

>* ReLU beschleunigt Training tiefer CNNs
>* Dropout stärkt robuste Verallgemeinerung



In [ ]:
#@title Python-Code - Rechenaufwand verstehen

# Dieses Beispiel schätzt Rechenaufwand kleiner CNN-Bausteine.
# Filtergröße, Filteranzahl und Pooling werden verglichen.
# Die Grafik zeigt, warum Architekturentscheidungen wichtig sind.

import numpy as np
import matplotlib.pyplot as plt

# Diese Funktion berechnet grob Multiplikationen einer Faltung.
def conv_multiplications(height, width, channels, filters, kernel_size):
    output_height = height - kernel_size + 1
    output_width = width - kernel_size + 1
    return output_height * output_width * channels * filters * kernel_size * kernel_size

# Wir prüfen eine kleine, synthetische RGB-Eingabe.
image_height = 64
image_width = 64
input_channels = 3

# Diese Varianten isolieren typische AlexNet-Ideen.
variant_names = ["3x3, 8 Filter", "5x5, 8 Filter", "5x5, 16 Filter", "Pooling danach"]
operation_counts = []

# Größere Filter und mehr Filter erhöhen den Aufwand.
operation_counts.append(conv_multiplications(64, 64, 3, 8, 3))
operation_counts.append(conv_multiplications(64, 64, 3, 8, 5))
operation_counts.append(conv_multiplications(64, 64, 3, 16, 5))

# Pooling halbiert Breite und Höhe vor der nächsten Faltung.
pooled_count = conv_multiplications(32, 32, 3, 16, 5)
operation_counts.append(pooled_count)

# Eine einfache Prüfung schützt vor unpassenden Eingabegrößen.
if min(image_height, image_width) < 5:
    raise ValueError("Das Beispielbild ist zu klein für den 5x5-Filter.")

# Wir geben wenige, gut lesbare Kennzahlen aus.
print("Grobe Multiplikationen pro Faltung:")
for name, count in zip(variant_names, operation_counts):
    print(f"{name}: {count / 1_000_000:.2f} Millionen")

# Die Balken machen den Unterschied sofort sichtbar.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(variant_names, np.array(operation_counts) / 1_000_000, color="steelblue")
ax.set_title("Rechenaufwand bei kleinen CNN-Entscheidungen")

# Achsenbeschriftungen erklären die gemessene Größe.
ax.set_xlabel("Architekturvariante")
ax.set_ylabel("Multiplikationen in Millionen")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()

plt.show()



## **2. Keras Applications**

### **2.1. Keras Applications Überblick**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_02_01.jpg?v=1787659451" width="250">



>* Vortrainierte Bildmodelle sparen aufwendiges Training
>* Transfer Learning nutzt gelernte visuelle Muster

>* MobileNetV2 ist leicht und CPU-freundlich
>* Es nutzt gelernte Bildmerkmale für neue Aufgaben

>* Vortrainierte Schichten bleiben unverändert
>* Kleiner Klassifikationskopf lernt ressourcenschonend



### **2.2. MobileNetV2 laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_02_02.jpg?v=1787659455" width="250">



>* Vortrainiertes MobileNetV2 erkennt allgemeine Bildmuster
>* Gelernte Merkmale für neue Aufgaben nutzen

>* Nur Faltungsteil als Merkmalsextraktor laden
>* Neuen Kopf für eigene Klassen ergänzen

>* Gewichte einfrieren spart Rechenzeit und Speicher.
>* Gelerntes Wissen bleibt für kleine Datensätze nutzbar.



In [ ]:
#@title Python-Code - MobileNetV2 laden

# Wir laden MobileNetV2 als eingefrorene Keras-Basis.
# Der Klassifikationskopf wird bewusst weggelassen.
# Die Ausgabe zeigt Formen und trainierbare Parameter.

import tensorflow as tf

# MobileNetV2 wird ohne ursprünglichen ImageNet-Kopf geladen.
base_model = tf.keras.applications.MobileNetV2(
    weights=None,
    include_top=False,
    input_shape=(96, 96, 3)
)

# Einfrieren verhindert Training der vielen Basisparameter.
base_model.trainable = False

# Ein kleiner Kopf passt die Merkmale an zwei Klassen an.
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(2, activation="softmax")
])

# Ein Beispielbild prüft die Form des Merkmalsextraktors.
dummy_images = tf.zeros((1, 96, 96, 3))
features = base_model(dummy_images, training=False)

# Die Zahlen machen den Transfer-Learning-Aufbau sichtbar.
base_params = base_model.count_params()
trainable_params = sum(
    tf.keras.backend.count_params(weight) for weight in model.trainable_weights
)

print("MobileNetV2 ohne Klassifikationskopf geladen.")
print(f"Eingabeform: {model.input_shape}")
print(f"Merkmalsform nach MobileNetV2: {features.shape}")
print(f"Parameter der eingefrorenen Basis: {base_params:,}")
print(f"Trainierbare Parameter im neuen Kopf: {trainable_params:,}")



### **2.3. Eingaben passend skalieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_02_03.jpg?v=1787659453" width="250">



>* Eingaben wie beim Vortraining vorbereiten
>* Eingefrorene Gewichte brauchen passende Skalierung

>* Bilder einheitlich auf MobileNetV2-Größe bringen
>* Pixelwerte passend normalisieren für starke Merkmale

>* Skalierung gehört fest in die Datenpipeline
>* Einheitliche Vorverarbeitung macht Transfer Learning fair



In [ ]:
#@title Python-Code - Eingaben passend skalieren

# Wir skalieren Eingaben für MobileNetV2 korrekt.
# Pixelwerte werden in den erwarteten Bereich gebracht.
# Die Ausgabe zeigt Form und Wertebereich.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Ein kleines synthetisches RGB-Bild ersetzt externe Bilddateien.
height = 96
width = 96
x_values = np.linspace(0, 255, width, dtype=np.float32)

# Farbkanäle erzeugen einfache, gut sichtbare Muster.
red_channel = np.tile(x_values, (height, 1))
green_channel = np.flipud(red_channel)
blue_channel = np.full((height, width), 128, dtype=np.float32)

# MobileNetV2 erwartet RGB-Bilder mit drei Farbkanälen.
image_uint8 = np.stack(
    [red_channel, green_channel, blue_channel], axis=-1
).astype(np.uint8)

# Für Keras-Modelle kommt zusätzlich eine Batch-Achse hinzu.
batch_uint8 = np.expand_dims(image_uint8, axis=0)

# Die passende Keras-Vorverarbeitung skaliert auf ungefähr minus eins bis eins.
preprocessed_batch = tf.keras.applications.mobilenet_v2.preprocess_input(
    batch_uint8.astype(np.float32)
)

# Eine einfache Prüfung macht die erwartete Eingabeform sichtbar.
expected_shape = (1, height, width, 3)
if batch_uint8.shape != expected_shape:
    raise ValueError("Die Bildform passt nicht zur erwarteten RGB-Batch-Form.")

# Kurze Ausgaben vergleichen Rohdaten und vorverarbeitete Daten.
print(f"Rohdaten-Form: {batch_uint8.shape}")
print(f"Rohdaten-Wertebereich: {batch_uint8.min()} bis {batch_uint8.max()}")
print(f"Skalierte Form: {preprocessed_batch.shape}")
print(
    "Skalierter Wertebereich: "
    f"{preprocessed_batch.min():.1f} bis "
    f"{preprocessed_batch.max():.1f}"
)

# Die Grafik zeigt das synthetische Bild vor der numerischen Skalierung.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image_uint8)
ax.set_title("Synthetisches RGB-Bild vor MobileNetV2-Skalierung")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



## **3. Transfer Learning Vergleich**

### **3.1. Basis einfrieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_03_01.jpg?v=1787659446" width="250">



>* Eingefrorene Basis nutzt vorhandenes Bildwissen
>* Kleines CNN muss Merkmale selbst lernen

>* Eingefrorene Basis macht Vergleiche klarer.
>* Kleine CNNs überfitten bei wenig Daten.

>* Einfrieren ist Vergleichsbasis, keine Erfolgsgarantie
>* Bei Bedarf obere Schichten feinjustieren



In [ ]:
#@title Python-Code - Basis einfrieren

# Wir frieren eine vortrainierte CNN-Basis ein.
# Nur der neue Klassifikationskopf bleibt trainierbar.
# Die Parameterzahlen machen den Vergleich sichtbar.

import tensorflow as tf
import matplotlib.pyplot as plt

# Diese Eingabegröße hält das Beispiel CPU-freundlich.
input_shape = (96, 96, 3)

# MobileNetV2 wird ohne Internetgewichte erstellt.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=input_shape,
    include_top=False,
    weights=None,
)

# Einfrieren bedeutet, dass Basisgewichte unverändert bleiben.
base_model.trainable = False

# Der Kopf lernt die neue Klassifikationsentscheidung.
model = tf.keras.Sequential(
    [
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(2, activation="softmax"),
    ]
)

# Ein Dummy-Aufruf baut alle Schichten vollständig auf.
dummy_images = tf.zeros((2, 96, 96, 3))
_ = model(dummy_images)

# Wir zählen trainierbare und eingefrorene Parameter.
trainable_params = int(
    sum(tf.keras.backend.count_params(weight) for weight in model.trainable_weights)
)

frozen_params = int(
    sum(tf.keras.backend.count_params(weight) for weight in model.non_trainable_weights)
)

# Diese Werte zeigen, welcher Modellteil wirklich lernt.
total_params = trainable_params + frozen_params
trainable_percent = 100 * trainable_params / total_params

print(f"TensorFlow-Version: {tf.__version__}")
print(f"Eingefrorene Basis: {frozen_params:,} Parameter")
print(f"Trainierbarer Kopf: {trainable_params:,} Parameter")
print(f"Trainierbarer Anteil: {trainable_percent:.3f}%")
print("Vergleichsidee: Transfer Learning trainiert hier nur den kleinen Kopf.")

# Das Balkendiagramm macht den Größenunterschied anschaulich.
labels = ["eingefrorene Basis", "trainierbarer Kopf"]
values = [frozen_params, trainable_params]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, values, color=["steelblue", "orange"])
ax.set_title("Basis einfrieren: Welche Parameter lernen?")
ax.set_xlabel("Modellteil")
ax.set_ylabel("Anzahl Parameter")
plt.show()



### **3.2. Klassifikationskopf trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_03_02.jpg?v=1787659450" width="250">



>* Eingefrorene Basis liefert allgemeine Bildmerkmale
>* Neuer Kopf ordnet Merkmale Klassen zu

>* Kleine CNNs lernen Merkmale komplett selbst
>* Transfer Learning lernt schneller mit weniger Daten

>* Lernkurven statt nur Genauigkeit bewerten
>* Domänenpassung mit kleinem CNN vergleichen



In [ ]:
#@title Python-Code - Klassifikationskopf trainieren

# Wir trainieren nur einen kleinen Klassifikationskopf.
# Eingefrorene Merkmale ersetzen hier eine CNN-Basis.
# Der Vergleich zeigt schnelleres Lernen durch Transfer.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

# Der Digits-Datensatz ist klein und offline verfügbar.
digits = load_digits()
images = digits.images
targets = digits.target

# Wir prüfen die wichtigste Formannahme vor dem Training.
if images.shape[0] != targets.shape[0]:
    raise ValueError("Bilder und Labels passen nicht zusammen.")

# Pixelwerte werden wie einfache eingefrorene Merkmale behandelt.
flat_pixels = images.reshape(images.shape[0], -1)
scaled_pixels = flat_pixels / 16.0

# Ein fester Split macht den Vergleich reproduzierbar.
X_train, X_test, y_train, y_test = train_test_split(
    scaled_pixels, targets, test_size=0.25, stratify=targets, random_state=42
)

# Dieses Modell steht für einen kleinen Kopf auf fertigen Merkmalen.
head_model = LogisticRegression(max_iter=300, solver="lbfgs", random_state=42)
head_model.fit(X_train, y_train)
head_accuracy = accuracy_score(y_test, head_model.predict(X_test))

# Dieses Modell muss zusätzliche Merkmalskombinationen selbst lernen.
small_cnn_proxy = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LogisticRegression(max_iter=300, solver="lbfgs", random_state=42),
)

small_cnn_proxy.fit(X_train[:250], y_train[:250])
proxy_accuracy = accuracy_score(y_test, small_cnn_proxy.predict(X_test))

# Die Ausgaben bleiben kurz und vergleichen beide Lernprobleme.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Kopf auf eingefrorenen Merkmalen: {head_accuracy:.3f}")
print(f"Kleines Modell mit wenig Training: {proxy_accuracy:.3f}")

# Ein Balkendiagramm macht den Unterschied sofort sichtbar.
labels = ["Transfer-Kopf", "Kleines CNN-Proxy"]
accuracies = [head_accuracy, proxy_accuracy]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, accuracies, color=["seagreen", "steelblue"])
ax.set_title("Validierungsgenauigkeit im Vergleich")
ax.set_xlabel("Ansatz")
ax.set_ylabel("Genauigkeit")
ax.set_ylim(0, 1)
plt.show()



### **3.3. Transfer Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_16/Lecture_B/image_03_03.jpg?v=1787659448" width="250">



>* Fairer Vergleich auf derselben Bildaufgabe
>* Vortrainierte Merkmale helfen bei kleinen Datensätzen

>* Kleine CNNs können schnell überanpassen
>* Transfer Learning umfassend und kritisch vergleichen

>* Fehler fachlich analysieren, nicht nur zählen
>* Modellwahl am Anwendungskontext begründen



In [ ]:
#@title Python-Code - Transfer Projekt

# Dieses Projekt vergleicht zwei Bildklassifikationsansätze.
# Ein kleines CNN lernt Merkmale selbst.
# Transfer Learning nutzt eingefrorene vortrainierte Merkmale.

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Wir verwenden kleine Ziffernbilder als schnelle Bildaufgabe.
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.mnist.load_data()

# Kleine Teilmengen halten das Beispiel CPU-freundlich.
train_count = 1200
test_count = 400

# Die Bilder werden für Keras skaliert und erweitert.
x_train = x_train_full[:train_count].astype("float32") / 255.0
y_train = y_train_full[:train_count]

x_test = x_test_full[:test_count].astype("float32") / 255.0
y_test = y_test_full[:test_count]

# MobileNetV2 erwartet RGB-Bilder mit mindestens 32 Pixeln.
x_train_rgb = np.repeat(x_train[..., np.newaxis], 3, axis=-1)
x_test_rgb = np.repeat(x_test[..., np.newaxis], 3, axis=-1)

x_train_rgb = tf.image.resize(x_train_rgb, (32, 32)).numpy()
x_test_rgb = tf.image.resize(x_test_rgb, (32, 32)).numpy()

# Eine einfache Prüfung macht die Datenannahme sichtbar.
if x_train_rgb.shape != (train_count, 32, 32, 3):
    raise ValueError("Die vorbereiteten Bilder haben eine unerwartete Form.")

# Feste Seeds machen den Vergleich reproduzierbarer.
tf.keras.utils.set_random_seed(42)

# Das kleine CNN lernt alle Filter von Grund auf.
small_cnn = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(16, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation="softmax"),
    ]
)

small_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Das Transfer-Modell friert allgemeine Bildmerkmale ein.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,
    weights="imagenet",
)

base_model.trainable = False

transfer_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(10, activation="softmax"),
    ]
)

transfer_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Beide Modelle sehen dieselben Trainingsdaten und Epochen.
small_history = small_cnn.fit(
    x_train_rgb,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2,
    verbose=0,
)

transfer_history = transfer_model.fit(
    x_train_rgb,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2,
    verbose=0,
)

# Die Testdaten wurden beim Training nicht verwendet.
small_test = small_cnn.evaluate(x_test_rgb, y_test, verbose=0)
transfer_test = transfer_model.evaluate(x_test_rgb, y_test, verbose=0)

print(f"TensorFlow-Version: {tf.__version__}")
print(f"Kleines CNN Testgenauigkeit: {small_test[1]:.3f}")
print(f"Transfer-Modell Testgenauigkeit: {transfer_test[1]:.3f}")

# Die Kurven zeigen den Lernweg, nicht nur das Endergebnis.
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(small_history.history["val_accuracy"], marker="o", label="Kleines CNN")
ax.plot(transfer_history.history["val_accuracy"], marker="o", label="Transfer")

ax.set_title("Validierungsgenauigkeit im fairen Vergleich")
ax.set_xlabel("Epoche")

ax.set_ylabel("Genauigkeit")
ax.set_ylim(0, 1)

ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Transfer mit Keras**</font>


In this lecture, you learned to:
- Erklären AlexNet-Ideen wie ReLU, größere Filter, Pooling und Dropout anhand kleiner Modelle. 
- Nutzen MobileNetV2 als CPU-freundlichen eingefrorenen Merkmalsextraktor. 
- Vergleichen Transfer-Learning-Ergebnisse mit kleinen selbst trainierten CNNs. 

In the next Module (Module 17), we will go over 'PyTorch Grundlagen'